<a href="https://colab.research.google.com/github/Ravi110296/MLOps-Ravi_Sharma-M25CSA024/blob/Ravi_Sharma_M25CSA024_lab2_worksheet/CNN_CIFAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install -q torch torchvision wandb thop matplotlib

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import wandb
from thop import profile

# ------------------ WandB Init ------------------
wandb.init(project="cifar10-cnn-lab")

# ------------------ Transforms ------------------
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# ------------------ Custom DataLoader ------------------
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=True, transform=transform)

trainloader = DataLoader(trainset, batch_size=128, shuffle=True)
testloader = DataLoader(testset, batch_size=128, shuffle=False)

# ------------------ CNN Model ------------------
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.fc = nn.Sequential(
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

model = SimpleCNN().cuda()

# ------------------ FLOPs Count ------------------
dummy_input = torch.randn(1, 3, 32, 32).cuda()
flops, params = profile(model, inputs=(dummy_input,))
print("FLOPs:", flops)
print("Params:", params)

wandb.log({"FLOPs": flops, "Params": params})

# ------------------ Training ------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(30):
    running_loss = 0.0
    correct = 0
    total = 0

    model.train()

    for images, labels in trainloader:
        images, labels = images.cuda(), labels.cuda()

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()

        # -------- Log gradients --------
        for name, param in model.named_parameters():
            if param.grad is not None:
                wandb.log({
                    f"gradients/{name}": wandb.Histogram(
                        param.grad.detach().cpu().numpy()
                    )
                })

        optimizer.step()
        running_loss += loss.item()

        # -------- Accuracy --------
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    avg_loss = running_loss / len(trainloader)

    # -------- Log to WandB --------
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_loss,
        "train_accuracy": train_acc
    })

    print(f"Epoch [{epoch+1}/30] Loss: {avg_loss:.4f}, Accuracy: {train_acc:.2f}%")

FLOPs,▁
Params,▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_loss,█▆▅▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
FLOPs,6654464.0
Params,1070794.0
epoch,29
train_loss,0.52675


[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
FLOPs: 6654464.0
Params: 1070794.0
Epoch [1/30] Loss: 1.5754, Accuracy: 42.68%
Epoch [2/30] Loss: 1.2361, Accuracy: 55.53%
Epoch [3/30] Loss: 1.0956, Accuracy: 61.18%
Epoch [4/30] Loss: 1.0001, Accuracy: 64.57%
Epoch [5/30] Loss: 0.9365, Accuracy: 66.95%
Epoch [6/30] Loss: 0.8815, Accuracy: 69.04%
Epoch [7/30] Loss: 0.8434, Accuracy: 70.43%
Epoch [8/30] Loss: 0.8027, Accuracy: 71.86%
Epoch [9/30] Loss: 0.7757, Accuracy: 72.86%
Epoch [10/30] Loss: 0.7475, Accuracy: 73.67%
Epoch [11/30] Loss: 0.7270, Accuracy: 74.55%
Epoch [12/30] Loss: 0.7021, Accuracy: 75.36%
Epoch [13/30] Loss: 0.6878, Accuracy: 76.04%
Epoch